In [0]:
CREATE OR REPLACE TABLE cfpb_risk.gold.issue_clusters_monthly AS
WITH monthly_base AS (
  SELECT
    institution_display_name, 
    rssd_id, 
    complaint_month, 
    product, 
    issue, 
    COUNT(*) AS complaint_count
  FROM cfpb_risk.silver.cfpb_complaints_clean
  GROUP BY 
    institution_display_name,
    rssd_id, 
    complaint_month, 
    product, 
    issue    
),
monthly_enriched AS (
  SELECT
    institution_display_name, 
    rssd_id, 
    complaint_month, 
    product, 
    issue, 
    complaint_count,
    LAG(complaint_count, 1)
      OVER (
        PARTITION BY 
          rssd_id,
          product, 
          issue
        ORDER BY complaint_month
      ) AS prior_month_complaint_count,
    AVG(complaint_count) OVER (
      PARTITION BY 
        rssd_id,
        product, 
        issue
      ORDER BY complaint_month
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS rolling_3_month_avg,
    SUM(complaint_count) OVER (
      PARTITION BY 
        rssd_id,
        complaint_month)
      AS bank_total_complaints_monthly
  FROM monthly_base
)
SELECT
  institution_display_name,
  rssd_id,
  complaint_month,
  product,
  issue,
  complaint_count,
  prior_month_complaint_count,
  rolling_3_month_avg,
  bank_total_complaints_monthly,
  complaint_count/NULLIF(bank_total_complaints_monthly,0) as share_of_bank_complaints
FROM monthly_enriched